In [0]:
import pandas as pd

csv = pd.read_csv("/Workspace/Users/azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com/energy_usage_cleaned.csv")
df = spark.createDataFrame(csv)

In [0]:
from pyspark.sql.functions import *

df = df.withColumn("usage_date", to_date(col("timestamp")))

In [0]:
daily_summary = df.groupBy("usage_date", "device_name", "room_name") \
    .agg(round(sum("energy_kwh"), 3).alias("daily_energy_kwh")) \
    .orderBy("usage_date", "room_name", "device_name")

display(daily_summary)

usage_date,device_name,room_name,daily_energy_kwh
2026-06-13,Air Purifier,Bedroom 2,0.124
2026-06-13,Table Lamp,Bedroom 2,0.09
2026-06-13,Microwave Oven,Kitchen,1.092
2026-06-13,Refrigerator,Kitchen,1.661
2026-06-13,Air Conditioner,Living Room,4.273
2026-06-13,Smart TV,Living Room,1.162
2026-06-13,Air Conditioner,Master Bedroom,6.624
2026-06-13,Ceiling Fan,Master Bedroom,0.409
2026-06-13,Desktop Computer,Study,1.005
2026-06-13,Printer,Study,0.131


In [0]:

weekly_summary = df.groupBy("device_name", "room_name") \
    .agg(
        round(sum("energy_kwh"), 3).alias("weekly_total_kwh"),
        round(sum("energy_kwh"), 3).alias("avg_reading_kwh")
    ) \
    .orderBy(col("weekly_total_kwh").desc())

display(weekly_summary)



device_name,room_name,weekly_total_kwh,avg_reading_kwh
Air Conditioner,Living Room,34.657,34.657
Air Conditioner,Master Bedroom,32.227,32.227
Refrigerator,Kitchen,9.437,9.437
Desktop Computer,Study,9.116,9.116
Microwave Oven,Kitchen,8.986,8.986
Smart TV,Living Room,6.922,6.922
Ceiling Fan,Master Bedroom,2.942,2.942
Table Lamp,Bedroom 2,1.934,1.934
Air Purifier,Bedroom 2,1.84,1.84
Printer,Study,0.736,0.736


In [0]:

device_daily_avg = daily_summary.groupBy("device_name") \
    .agg(round(avg("daily_energy_kwh"), 3).alias("device_avg_daily_kwh"))

savings_check = daily_summary.join(device_daily_avg, on="device_name") \
    .withColumn(
        "savings_flag",
        when(col("daily_energy_kwh") > col("device_avg_daily_kwh") * 1.3, "High Usage - Review")
        .otherwise("Normal")
    )

savings_opportunities = savings_check.filter(col("savings_flag") == "High Usage - Review") \
    .select("usage_date", "device_name", "room_name", "daily_energy_kwh", "device_avg_daily_kwh", "savings_flag") \
    .orderBy(col("daily_energy_kwh").desc())

display(savings_opportunities)


usage_date,device_name,room_name,daily_energy_kwh,device_avg_daily_kwh,savings_flag
2026-06-13,Air Conditioner,Master Bedroom,6.624,4.777,High Usage - Review
2026-06-14,Air Conditioner,Living Room,6.541,4.777,High Usage - Review
2026-06-17,Desktop Computer,Study,2.097,1.302,High Usage - Review
2026-06-16,Desktop Computer,Study,1.88,1.302,High Usage - Review
2026-06-19,Smart TV,Living Room,1.298,0.989,High Usage - Review
2026-06-14,Ceiling Fan,Master Bedroom,0.61,0.42,High Usage - Review
2026-06-17,Ceiling Fan,Master Bedroom,0.563,0.42,High Usage - Review
2026-06-16,Air Purifier,Bedroom 2,0.429,0.263,High Usage - Review


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS smart_home_energy")

daily_summary.write.format("delta").mode("overwrite").saveAsTable("smart_home_energy.daily_summary")

weekly_summary.write.format("delta").mode("overwrite").saveAsTable("smart_home_energy.weekly_summary")

savings_opportunities.write.format("delta").mode("overwrite").saveAsTable("smart_home_energy.savings_opportunities")

In [0]:
daily_pd = spark.table("az_dl.smart_home_energy.daily_summary").toPandas()

daily_pd.to_csv(
    "/Workspace/Users/azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com/daily_summary.csv",
    index=False
)

weekly_pd = spark.table("az_dl.smart_home_energy.weekly_summary").toPandas()

weekly_pd.to_csv(
    "/Workspace/Users/azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com/weekly_summary.csv",
    index=False
)

savings_pd = spark.table("az_dl.smart_home_energy.savings_opportunities").toPandas()

savings_pd.to_csv(
    "/Workspace/Users/azuser7230_mml.local@karthikirisoutlook.onmicrosoft.com/savings_opportunities.csv",
    index=False
)


In [0]:
daily_summary.createOrReplaceTempView("daily_summary_view")
weekly_summary.createOrReplaceTempView("weekly_summary_view")

In [0]:
spark.sql("""
SELECT device_name, room_name, weekly_total_kwh
FROM weekly_summary_view
WHERE weekly_total_kwh > (
    SELECT AVG(weekly_total_kwh)
    FROM weekly_summary_view
)
ORDER BY weekly_total_kwh DESC
""").show(truncate=False)

+---------------+--------------+----------------+
|device_name    |room_name     |weekly_total_kwh|
+---------------+--------------+----------------+
|Air Conditioner|Living Room   |34.657          |
|Air Conditioner|Master Bedroom|32.227          |
+---------------+--------------+----------------+



In [0]:
spark.sql("""
SELECT room_name,
       ROUND(SUM(daily_energy_kwh), 3) AS room_weekly_total
FROM daily_summary_view
GROUP BY room_name
ORDER BY room_weekly_total DESC
""").show(truncate=False)

+--------------+-----------------+
|room_name     |room_weekly_total|
+--------------+-----------------+
|Living Room   |41.579           |
|Master Bedroom|35.17            |
|Kitchen       |18.423           |
|Study         |9.853            |
|Bedroom 2     |3.776            |
+--------------+-----------------+

